In [147]:
import pandas as pd
import duckdb

pd.set_option('display.max_columns', 50)

## Functions

In [148]:
import plotly.graph_objects as go

def plot_events_over_time(df, title=''):

    fig = go.Figure()

    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["num_competitions"],
                mode="lines+markers",
                name=event_name,
                line=dict(width=3),
                marker=dict(size=8),
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Ano: %{x}<br>"
                    "Competições: %{y}<extra></extra>"
                )
            )
        )

    total_df = (
        df[["year", "total_competitions"]]
        .drop_duplicates()
        .sort_values("year")
    )

    fig.add_trace(
        go.Scatter(
            x=total_df["year"],
            y=total_df["total_competitions"],
            mode="lines+markers",
            name="Total de Competições",
            line=dict(width=6),
            marker=dict(size=10),
            hovertemplate=(
                "<b>Total de Competições</b><br>"
                "Ano: %{x}<br>"
                "Total: %{y}<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Ano",
        yaxis_title="Número de Competições",
        hovermode="closest",
        template="plotly_white",
        legend_title="Modalidade",
        height=700,
        legend=dict(
            itemclick="toggle",
            itemdoubleclick="toggleothers"
        )
    )

    return fig

def plot_event_presence_pct(df, title=''):

    fig = go.Figure()

    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["presence_pct"] * 100,
                mode="lines+markers",
                name=event_name,
                line=dict(width=3),
                marker=dict(size=8),
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Ano: %{x}<br>"
                    "Presença: %{y:.1f}%<extra></extra>"
                )
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Ano",
        yaxis_title="% das Competições",
        hovermode="closest",
        template="plotly_white",
        legend_title="Modalidade",
        height=700,
        legend=dict(
            itemclick="toggle",
            itemdoubleclick="toggleothers"
        )
    )

    return fig

## Análise mundo

In [149]:
con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,974
1,222,2x2x2 Cube,792
2,pyram,Pyraminx,698
3,444,4x4x4 Cube,691
4,skewb,Skewb,594
5,333oh,3x3x3 One-Handed,592
6,clock,Clock,564
7,555,5x5x5 Cube,527
8,minx,Megaminx,514
9,333bf,3x3x3 Blindfolded,504


In [150]:
df_camp_by_year = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT c.id) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
WHERE
    c.cancelled = 0
GROUP BY c.year
ORDER BY c.year
""").df()

In [151]:
df_final = pd.DataFrame()
for i in range(2003, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [152]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [153]:
fig_absolute_world = plot_events_over_time(df_final, 'Quantidade de Competições por Modalidade no Mundo')
fig_absolute_world

In [154]:
fig_percentage_world = plot_event_presence_pct(df_final, 'Presença das Modalidades nas Competições Mundiais')
fig_percentage_world

## Brasil

In [155]:
con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.country_id = 'Brazil'
    AND c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,38
1,222,2x2x2 Cube,34
2,pyram,Pyraminx,30
3,minx,Megaminx,29
4,clock,Clock,29
5,444,4x4x4 Cube,28
6,333oh,3x3x3 One-Handed,25
7,skewb,Skewb,24
8,333bf,3x3x3 Blindfolded,22
9,555,5x5x5 Cube,21


In [156]:
df_camp_by_year = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT c.id) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
WHERE
    c.country_id = 'Brazil'
    AND c.cancelled = 0
GROUP BY c.year
ORDER BY c.year
""").df()

In [157]:
df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.country_id = 'Brazil'
        AND c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [158]:
df_final.head()

,event_id,event_name,num_competitions,year,total_competitions
0,444,4x4x4 Cube,1,2007,1
1,333,3x3x3 Cube,1,2007,1
2,555,5x5x5 Cube,1,2007,1
3,222,2x2x2 Cube,1,2007,1
4,minx,Megaminx,1,2007,1


In [159]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [160]:
fig_absolute_brazil = plot_events_over_time(df_final, 'Quantidade de Competições por Modalidade no Brasil')
fig_absolute_brazil

In [161]:
fig_percentage_brazil = plot_event_presence_pct(df_final, 'Presença das Modalidades nas Competições Brasileiras')
fig_percentage_brazil

## SP

In [162]:
con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.country_id = 'Brazil'
    AND c.city_name LIKE '%, São Paulo'
    AND c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,9
1,clock,Clock,8
2,333bf,3x3x3 Blindfolded,7
3,minx,Megaminx,6
4,222,2x2x2 Cube,6
5,333oh,3x3x3 One-Handed,6
6,444,4x4x4 Cube,6
7,pyram,Pyraminx,6
8,777,7x7x7 Cube,6
9,666,6x6x6 Cube,5


In [163]:
df_camp_by_year = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT c.id) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
WHERE
    c.country_id = 'Brazil'
    AND c.city_name LIKE '%, São Paulo'
    AND c.cancelled = 0
GROUP BY c.year
ORDER BY c.year
""").df()

In [164]:
df_camp_by_year

,year,total_competitions
0,2007,1
1,2009,2
2,2010,2
3,2011,1
4,2012,3
5,2013,9
6,2014,9
7,2015,11
8,2016,9
9,2017,14


In [165]:
df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.country_id = 'Brazil'
        AND c.city_name LIKE '%, São Paulo'
        AND c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [166]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [167]:
fig_absolute_sp = plot_events_over_time(df_final, 'Quantidade de Competições por Modalidade Em São Paulo')
fig_absolute_sp

In [168]:
fig_percentage_sp = plot_event_presence_pct(df_final, 'Presença das Modalidades nas Competições Paulistas')
fig_percentage_sp

## Rounds

In [169]:
df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
        WITH rounds AS (

            SELECT DISTINCT
                competition_id,
                event_id,
                round_type_id,
                format_id

            FROM read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t')

        )

        SELECT
            e.id AS event_id,
            e.name AS event_name,

            SUM(
                CASE

                    WHEN e.id IN ('333fm', '333mbf')
                    THEN
                        CASE rounds.format_id
                            WHEN '1' THEN 1
                            WHEN '2' THEN 2
                            WHEN '3' THEN 3
                            ELSE 1
                        END

                    ELSE 1

                END
            ) AS competitive_units

        FROM rounds

        JOIN read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
            ON c.id = rounds.competition_id

        JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
            ON e.id = rounds.event_id

        WHERE
            c.country_id = 'Brazil'
            AND c.city_name LIKE '%, São Paulo'
            AND c.year = {i}

        GROUP BY
            e.id,
            e.name

        ORDER BY
            competitive_units DESC
        """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [170]:
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [171]:
fig_rounds_sp = plot_events_over_time(df_final.rename(columns={'competitive_units': 'num_competitions'}), 'Quantidade de rounds por Modalidade Em São Paulo')
fig_rounds_sp

## HTML Final

In [172]:
from plotly.io import to_html

def export_dashboard(figures, output_file="index.html"):
    """
    figures: list[go.Figure]
    """

    chart_blocks = []

    for i, fig in enumerate(figures):

        graph_html = to_html(
            fig,
            include_plotlyjs="cdn" if i == 0 else False,
            full_html=False
        )

        title = fig.layout.title.text or f"Gráfico {i + 1}"

        chart_blocks.append(
            f"""
            <div class="chart">
                <h2>{title}</h2>
                {graph_html}
            </div>
            """
        )

    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">

        <title>WCA Brasil - Análise de Modalidades</title>

        <style>
            body {{
                font-family: Arial, sans-serif;
                max-width: 1400px;
                margin: auto;
                padding: 20px;
            }}

            h1 {{
                text-align: center;
            }}

            .chart {{
                margin-top: 50px;
                margin-bottom: 50px;
            }}
        </style>
    </head>

    <body>

        <h1>
            Evolução das Modalidades da WCA
        </h1>

        <p>
            Análise baseada nos
            <a href="https://www.worldcubeassociation.org/export/results"
               target="_blank"
               rel="noopener noreferrer">
               dados públicos da WCA
            </a>.
        </p>

        {"".join(chart_blocks)}

    </body>
    </html>
    """

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)

In [173]:
export_dashboard([fig_absolute_world, fig_percentage_world, fig_absolute_brazil, fig_percentage_brazil, fig_absolute_sp, fig_percentage_sp, fig_rounds_sp])